# MusicBrainz French Artist Registry

*Disclaimer*: this version of the notebook has been adapted to run in the Google Colab environment

## 1) Environment and Data Access

Mounting storage, initializing paths, and configuring imports/utilities to read MusicBrainz dump tables.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

candidate_dirs = [
    Path.cwd() / 'data/dataset/mbdump',
    Path.cwd().parent / 'data/dataset/mbdump',
    Path('..') / 'data/dataset/mbdump',
    Path('/content/drive/MyDrive/masters/thesis_french_song_nlp/data/dataset/mbdump'),
]
DATA_DIR = next((p for p in candidate_dirs if p.exists()), None)
assert DATA_DIR is not None, 'Missing path: data/dataset/mbdump not found from current context'
print('Using data from:', DATA_DIR.resolve())

Using data from: /content/drive/MyDrive/masters/thesis_french_song_nlp/data/dataset/mbdump


In [3]:
def read_mb(filename, names, usecols=None, dtype='string'):
    path = DATA_DIR / filename
    return pd.read_csv(
        path,
        sep='\t',
        names=names,
        header=None,
        usecols=usecols,
        dtype=dtype,
        na_values='\\N',
        keep_default_na=True,
        low_memory=False,
    )

## 2) Core MusicBrainz Table Loading

Load foundational entities (artists, releases, recordings, credits, languages, and metadata) used for feature assembly.

Base tables

In [ ]:
track = read_mb('track', [
    'track_id', 'track_gid', 'recording_id', 'medium_id', 'position', 'number',
    'track_name', 'track_artist_credit_id', 'track_length_ms', 'edits_pending',
    'last_updated', 'is_data_track'
], usecols=[0, 2, 3, 6, 7, 8])

medium = read_mb('medium', [
    'medium_id', 'release_id', 'position', 'format_id', 'name',
    'edits_pending', 'last_updated', 'track_count', 'medium_gid'
], usecols=[0, 1])

release = read_mb('release', [
    'release_id', 'release_gid', 'release_name', 'release_artist_credit_id',
    'release_group_id', 'status_id', 'packaging_id', 'release_language_id',
    'release_script_id', 'barcode', 'comment', 'edits_pending', 'quality', 'last_updated'
], usecols=[0, 2, 3, 7])

release_country = read_mb('release_country', [
    'release_id', 'country_area_id', 'year', 'month', 'day'
], usecols=[0, 1, 2, 3, 4])

In [5]:
recording = read_mb('recording', [
    'recording_id', 'recording_gid', 'recording_name', 'recording_artist_credit_id',
    'recording_length_ms', 'comment', 'edits_pending', 'last_updated', 'is_video'
], usecols=[0, 2, 3, 4, 8])

In [6]:
artist_credit = read_mb('artist_credit', [
    'artist_credit_id', 'artist_credit_name', 'artist_count', 'ref_count',
    'last_updated', 'edits_pending', 'artist_credit_gid'
], usecols=[0, 1])

artist_credit_name = read_mb('artist_credit_name', [
    'artist_credit_id', 'position', 'artist_id', 'name', 'join_phrase'
], usecols=[0, 1, 2])

artist = read_mb('artist', [
    'artist_id', 'artist_gid', 'artist_name', 'sort_name', 'begin_date_year',
    'begin_date_month', 'begin_date_day', 'end_date_year', 'end_date_month',
    'end_date_day', 'artist_type_id', 'artist_area_id', 'gender_id', 'comment',
    'edits_pending', 'last_updated', 'ended', 'begin_area_id', 'end_area_id'
], usecols=[0, 2, 11, 12])

In [7]:
gender = read_mb('gender', ['gender_id', 'gender_name', 'p3', 'p4', 'description', 'gender_gid'], usecols=[0, 1])
language = read_mb('language', ['language_id', 'iso_code_3', 'iso_code_3b', 'iso_code_1', 'language_name', 'frequency', 'iso_code_3t'], usecols=[0, 4, 6])
area = read_mb('area', ['area_id', 'area_gid', 'area_name', 'area_type_id', 'edits_pending', 'last_updated', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'ended', 'comment'], usecols=[0, 2, 3])
iso_3166_1 = read_mb('iso_3166_1', ['area_id', 'country_code'], usecols=[0, 1])

Genre and influence tables

In [ ]:
l_artist_genre = read_mb('l_artist_genre', ['id', 'link_id', 'artist_id', 'genre_id', 'edits_pending', 'last_updated', 'link_order', 'entity0_credit', 'entity1_credit'], usecols=[1, 2, 3])
genre = read_mb('genre', ['genre_id', 'genre_gid', 'genre_name', 'comment', 'edits_pending', 'last_updated'], usecols=[0, 2])

l_artist_artist = read_mb('l_artist_artist', ['id', 'link_id', 'artist0_id', 'artist1_id', 'edits_pending', 'last_updated', 'link_order', 'entity0_credit', 'entity1_credit'], usecols=[1, 2, 3])
link = read_mb('link', ['link_id', 'link_type_id', 'b0', 'b1', 'b2', 'b3', 'b4', 'b5', 'edits_pending', 'last_updated', 'ended'], usecols=[0, 1])
link_type = read_mb('link_type', [
    'link_type_id', 'parent_link_type_id', 'child_order', 'gid', 'entity0_type',
    'entity1_type', 'link_name', 'description', 'link_phrase', 'reverse_link_phrase',
    'long_link_phrase', 'last_updated', 'is_deprecated', 'has_dates', 'entity0_cardinality', 'entity1_cardinality'
], usecols=[0, 1, 4, 5, 6])

print('Loaded base MB tables')

Loaded base MB tables


## 3) Helper Mapping Construction

Build country, artist, genre, and influence mappings that support candidate scoring and downstream filtering.

In [ ]:
country_map = area[['area_id', 'area_name']].merge(iso_3166_1, on='area_id', how='left')
country_map = country_map.rename(columns={'area_name': 'country_name'})

release_country_enriched = release_country.merge(country_map[['area_id', 'country_code', 'country_name']], left_on='country_area_id', right_on='area_id', how='left')
release_country_enriched = release_country_enriched.drop(columns=['area_id'])

artist_primary = artist_credit_name[artist_credit_name['position'] == '0'][['artist_credit_id', 'artist_id']].drop_duplicates('artist_credit_id')

artist_meta = artist[['artist_id', 'artist_name', 'artist_area_id', 'gender_id']].merge(gender, on='gender_id', how='left')
artist_meta = artist_meta.merge(country_map[['area_id', 'country_code', 'country_name']], left_on='artist_area_id', right_on='area_id', how='left')
artist_meta = artist_meta.rename(columns={'country_code': 'artist_country_code', 'country_name': 'artist_country_name'})
artist_meta = artist_meta.drop(columns=['area_id'])

Artist genres (aggregated)

In [ ]:
artist_genres = l_artist_genre.merge(genre, on='genre_id', how='left')
artist_genres = artist_genres.dropna(subset=['genre_name'])
artist_genres = artist_genres.groupby('artist_id', as_index=False)['genre_name'].agg(lambda x: ' | '.join(sorted(set(x.astype(str)))))
artist_genres = artist_genres.rename(columns={'genre_name': 'artist_genres'})

Artist influence relations (best-effort): keep artist-artist relations with musical/influence semantics

In [ ]:
artist_links = l_artist_artist.merge(link, on='link_id', how='left').merge(link_type, on='link_type_id', how='left')
artist_links = artist_links[(artist_links['entity0_type'] == 'artist') & (artist_links['entity1_type'] == 'artist')]

name_pat = re.compile(r'influenc|teacher|collaboration|tribute|supporting|subgroup|member of band|founder|artistic director|rename', re.IGNORECASE)
artist_links = artist_links[
    artist_links['link_name'].fillna('').str.contains(name_pat)
    | (artist_links['parent_link_type_id'] == '106')
]

artist_links_named = artist_links.merge(artist[['artist_id', 'artist_name']], left_on='artist1_id', right_on='artist_id', how='left')
artist_links_named['edge'] = artist_links_named['link_name'].fillna('relation') + ':' + artist_links_named['artist_name'].fillna('unknown')
artist_influence = artist_links_named.groupby('artist0_id', as_index=False)['edge'].agg(lambda x: ' | '.join(sorted(set(x.astype(str)))))
artist_influence = artist_influence.rename(columns={'artist0_id': 'artist_id', 'edge': 'artist_influence_edges'})

print('Mappings built')

Mappings built


## 4) Stage 1: Candidate Artist Pool

Create a broad candidate registry with joined artist attributes and derived NLP/network-ready features.

Stage 1: candidate pool

Join chain: track -> medium -> release -> release_country + recording + artist credit + artist metadata

In [ ]:
song = track.merge(medium, on='medium_id', how='left')
song = song.merge(release, on='release_id', how='left')
song = song.merge(recording[['recording_id', 'recording_name', 'recording_artist_credit_id', 'recording_length_ms', 'is_video']], on='recording_id', how='left')
song = song.merge(release_country_enriched[['release_id', 'year', 'month', 'day', 'country_code', 'country_name']], on='release_id', how='left')
song = song.merge(language[['language_id', 'language_name', 'iso_code_3t']], left_on='release_language_id', right_on='language_id', how='left')

Resolve artist credit name


In [ ]:
song = song.merge(artist_credit[['artist_credit_id', 'artist_credit_name']], left_on='track_artist_credit_id', right_on='artist_credit_id', how='left')
song = song.rename(columns={'artist_credit_name': 'track_credit_name'})
song = song.drop(columns=['artist_credit_id'])

song = song.merge(artist_credit[['artist_credit_id', 'artist_credit_name']], left_on='recording_artist_credit_id', right_on='artist_credit_id', how='left')
song = song.rename(columns={'artist_credit_name': 'recording_credit_name'})
song = song.drop(columns=['artist_credit_id'])

song['artist_credit_name'] = song['track_credit_name'].combine_first(song['recording_credit_name'])
song = song.drop(columns=['track_credit_name', 'recording_credit_name'])

Link primary artist id from chosen credit id

In [ ]:
song['credit_id_for_artist'] = song['track_artist_credit_id'].combine_first(song['recording_artist_credit_id'])
song = song.merge(artist_primary, left_on='credit_id_for_artist', right_on='artist_credit_id', how='left')
song = song.drop(columns=['artist_credit_id'])

song = song.merge(artist_meta, on='artist_id', how='left')
song = song.merge(artist_genres, on='artist_id', how='left')
song = song.merge(artist_influence, on='artist_id', how='left')

song['song_title'] = song['track_name'].combine_first(song['recording_name'])
song['release_country_code'] = song['country_code']
song['release_country_name'] = song['country_name']

Additional NLP / network features

In [ ]:
song['title_len_chars'] = song['song_title'].fillna('').str.len()
song['title_token_count'] = song['song_title'].fillna('').str.split().str.len()
song['has_non_latin_chars'] = song['song_title'].fillna('').str.contains(r'[^\x00-\x7F]', regex=True)
song['artist_influence_edge_count'] = song['artist_influence_edges'].fillna('').str.count(r'\|') + np.where(song['artist_influence_edges'].notna(), 1, 0)
song.loc[song['artist_influence_edges'].isna(), 'artist_influence_edge_count'] = 0

candidate_columns = [
    'year', 'month', 'day', 'song_title', 'artist_credit_name', 'artist_name',
    'language_name', 'iso_code_3t', 'release_country_code', 'release_country_name',
    'artist_country_code', 'artist_country_name', 'gender_name', 'artist_genres',
    'artist_influence_edges', 'title_len_chars', 'title_token_count',
    'has_non_latin_chars', 'track_length_ms', 'recording_length_ms', 'is_video',
    'track_id', 'recording_id', 'release_id', 'artist_id'
]

candidates_stage1 = song[candidate_columns].copy()
candidates_stage1 = candidates_stage1.dropna(subset=['song_title'])

print('Stage 1 candidate rows:', len(candidates_stage1))
candidates_stage1.head(10)

Stage 1 candidate rows: 121327699


,year,month,day,song_title,artist_credit_name,artist_name,language_name,iso_code_3t,release_country_code,release_country_name,artist_country_code,artist_country_name,gender_name,artist_genres,artist_influence_edges,title_len_chars,title_token_count,has_non_latin_chars,track_length_ms,recording_length_ms,is_video,track_id,recording_id,release_id,artist_id
0,1995,<NA>,<NA>,The Ghost of Tom Joad,Bruce Springsteen,Bruce Springsteen,English,eng,ES,Spain,US,United States,Male,<NA>,collaboration:Artists United Against Apartheid...,21,5,False,263000,263000,f,34228823,428644,2990206,813
1,1991,<NA>,<NA>,Five Man Army,Massive Attack,Massive Attack,English,eng,CA,Canada,GB,United Kingdom,<NA>,<NA>,vocal supporting musician:Damon Reece,13,3,False,364306,364306,f,81,11,600623,4
2,2019,11,8,Wonder Girl,Sparks,Sparks,English,eng,XE,Europe,US,United States,<NA>,<NA>,collaboration:FFS,11,2,False,139000,139000,f,35831997,25849634,3153838,14389
3,1991,8,6,Five Man Army,Massive Attack,Massive Attack,English,eng,GB,United Kingdom,GB,United Kingdom,<NA>,<NA>,vocal supporting musician:Damon Reece,13,3,False,364306,364306,f,99,11,600626,4
4,1998,4,6,Five Man Army,Massive Attack,Massive Attack,English,eng,GB,United Kingdom,GB,United Kingdom,<NA>,<NA>,vocal supporting musician:Damon Reece,13,3,False,364306,364306,f,108,11,600627,4
5,2006,10,11,Five Man Army,Massive Attack,Massive Attack,English,eng,JP,Japan,GB,United Kingdom,<NA>,<NA>,vocal supporting musician:Damon Reece,13,3,False,364306,364306,f,126,11,825915,4
6,1991,8,6,Five Man Army,Massive Attack,Massive Attack,English,eng,US,United States,GB,United Kingdom,<NA>,<NA>,vocal supporting musician:Damon Reece,13,3,False,364306,364306,f,45,11,1275064,4
7,1991,<NA>,<NA>,Be Thankful for What You've Got,Massive Attack,Massive Attack,English,eng,CA,Canada,GB,United Kingdom,<NA>,<NA>,vocal supporting musician:Damon Reece,31,6,False,249693,249693,f,80,14,600623,4
8,2019,11,8,Girl From Germany,Sparks,Sparks,English,eng,XE,Europe,US,United States,<NA>,<NA>,collaboration:FFS,17,3,False,210000,210000,f,35831998,25849636,3153838,14389
9,1991,8,6,Be Thankful for What You've Got,Massive Attack,Massive Attack,English,eng,GB,United Kingdom,GB,United Kingdom,<NA>,<NA>,vocal supporting musician:Damon Reece,31,6,False,249693,249693,f,98,14,600626,4


## 5) Stage 2: Likely French Artist Selection

Apply strict France-oriented filters to retain artists likely associated with French repertoire context.

In [ ]:
# Strong signal: explicit French language tag
is_lang_fra = candidates_stage1['iso_code_3t'].fillna('').str.lower().eq('fra')

# Country-France signals
is_release_fr = candidates_stage1['release_country_code'].fillna('').str.upper().eq('FR')
is_artist_fr = candidates_stage1['artist_country_code'].fillna('').str.upper().eq('FR')

# Weak title signal (best-effort lexical cue)
fr_title_pattern = re.compile(r"\b(le|la|les|des|de|du|un|une|et|amour|coeur|vie|nuit|bonjour|soir|paris)\b", re.IGNORECASE)
is_french_like_title = candidates_stage1['song_title'].fillna('').str.contains(fr_title_pattern)

likely_score = (
    is_lang_fra.astype(int) * 3
    + is_release_fr.astype(int) * 2
    + is_artist_fr.astype(int) * 2
    + is_french_like_title.astype(int)
)

candidates_stage1['likely_french_score'] = likely_score
candidates_stage1['signal_language_fra'] = is_lang_fra
candidates_stage1['signal_release_fr'] = is_release_fr
candidates_stage1['signal_artist_fr'] = is_artist_fr
candidates_stage1['signal_title_french_like'] = is_french_like_title

france_country_subset = candidates_stage1[is_release_fr | is_artist_fr].copy()

likely_french_stage2 = france_country_subset[
    (france_country_subset['likely_french_score'] >= 3)
].copy()

print('France country subset rows:', len(france_country_subset))
print('Likely French (score >= 3) rows:', len(likely_french_stage2))
likely_french_stage2.sort_values(['likely_french_score', 'year'], ascending=[False, False]).head(20)

/tmp/ipykernel_1644/2823718640.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_french_like_title = candidates_stage1['song_title'].fillna('').str.contains(fr_title_pattern)


France country subset rows: 6892156
Likely French (score >= 3) rows: 4186516


,year,month,day,song_title,artist_credit_name,artist_name,language_name,iso_code_3t,release_country_code,release_country_name,artist_country_code,artist_country_name,gender_name,artist_genres,artist_influence_edges,title_len_chars,title_token_count,has_non_latin_chars,track_length_ms,recording_length_ms,is_video,track_id,recording_id,release_id,artist_id,likely_french_score,signal_language_fra,signal_release_fr,signal_artist_fr,signal_title_french_like
5102915,2026,2,20,Kyoto Dans La Brume,Airelle Besson & Lionel Suarez,Airelle Besson,French,fra,FR,France,FR,France,Female,<NA>,member of band:Rockingchair,19,4,False,242321,242321,f,56184192,44952089,5530524,1185830,8,True,True,True,True
5110285,2026,2,20,La Course,Airelle Besson & Lionel Suarez,Airelle Besson,French,fra,FR,France,FR,France,Female,<NA>,member of band:Rockingchair,9,2,False,202176,202176,f,56184195,44952092,5530524,1185830,8,True,True,True,True
5110721,2026,2,20,Le Jour J À L'Heure H,Airelle Besson & Lionel Suarez,Airelle Besson,French,fra,FR,France,FR,France,Female,<NA>,member of band:Rockingchair,21,6,True,288668,288668,f,56184198,44952095,5530524,1185830,8,True,True,True,True
5111311,2026,2,20,De Passage,Airelle Besson & Lionel Suarez,Airelle Besson,French,fra,FR,France,FR,France,Female,<NA>,member of band:Rockingchair,10,2,False,80459,80459,f,56184200,44952097,5530524,1185830,8,True,True,True,True
5112289,2026,2,20,Les Tuiles Bleues,Airelle Besson & Lionel Suarez,Airelle Besson,French,fra,FR,France,FR,France,Female,<NA>,member of band:Rockingchair,17,3,False,256127,256127,f,56184201,44952098,5530524,1185830,8,True,True,True,True
6152454,2026,1,22,Sous les caméras,Dilezenn,Dilezenn,French,fra,FR,France,FR,France,<NA>,<NA>,<NA>,16,3,True,<NA>,<NA>,f,56186870,44953811,5530909,3201877,8,True,True,True,True
6152460,2026,1,22,La neige dans le biberon vide,Dilezenn,Dilezenn,French,fra,FR,France,FR,France,<NA>,<NA>,<NA>,29,6,False,<NA>,<NA>,f,56186871,44953812,5530909,3201877,8,True,True,True,True
6152849,2026,1,22,Dans le noir,Dilezenn,Dilezenn,French,fra,FR,France,FR,France,<NA>,<NA>,<NA>,12,3,False,<NA>,<NA>,f,56186875,44953816,5530909,3201877,8,True,True,True,True
15476745,2026,1,30,Être et ne plus être,Lazuli,Lazuli,French,fra,FR,France,FR,France,<NA>,<NA>,<NA>,20,5,True,312000,312000,f,56242272,44992982,5538629,276078,8,True,True,True,True
15478421,2026,1,30,Une chanson cherokee,Lazuli,Lazuli,French,fra,FR,France,FR,France,<NA>,<NA>,<NA>,20,3,False,234000,234000,f,56242277,44992987,5538629,276078,8,True,True,True,True


## 6) Export and Derived Outputs

Persist stage outputs and create long-format genre expansions for downstream analytics and modeling.

In [17]:
out_dir = Path('data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

candidates_stage1.to_csv(out_dir / 'mb_candidates_stage1.tsv', sep='\t', index=False)
france_country_subset.to_csv(out_dir / 'mb_france_country_subset.tsv', sep='\t', index=False)
likely_french_stage2.to_csv(out_dir / 'mb_likely_french_stage2.tsv', sep='\t', index=False)

print('Saved:')
print('-', (out_dir / 'mb_candidates_stage1.tsv').resolve())
print('-', (out_dir / 'mb_france_country_subset.tsv').resolve())
print('-', (out_dir / 'mb_likely_french_stage2.tsv').resolve())

Saved:
- /content/data/processed/mb_candidates_stage1.tsv
- /content/data/processed/mb_france_country_subset.tsv
- /content/data/processed/mb_likely_french_stage2.tsv


In [18]:
artist_genre_long = (
    l_artist_genre[['artist_id', 'genre_id']]
    .merge(genre[['genre_id', 'genre_name']], on='genre_id', how='left')
    .dropna(subset=['genre_name'])
    .drop_duplicates()
)

artist_genre_long.head()

,artist_id,genre_id,genre_name
0,34423,94,d-beat
1,3221251,86,country
2,435717,15,ambient
3,435717,145,experimental
4,435717,204,idm


In [19]:
likely_french_with_genre = (
    likely_french_stage2
    .merge(artist_genre_long, on='artist_id', how='left')
)

likely_french_genre_agg = (
    likely_french_with_genre
    .groupby(['track_id', 'song_title', 'artist_name'], as_index=False)['genre_name']
    .agg(lambda x: ' | '.join(sorted(set(x.dropna().astype(str)))))
    .rename(columns={'genre_name': 'matched_genres'})
)

likely_french_genre_agg.head(20)

,track_id,song_title,artist_name,matched_genres
0,10000116,Kontraul,Yuksek,
1,10000117,Contact,Yuksek,
2,10000118,It Comes,Yuksek,
3,10000119,It Comes (extended),Yuksek,
4,10000494,Clarinet a La King,Benny Goodman and His Orchestra,
5,10000507,Clarinet a La King,Benny Goodman and His Orchestra,
6,10000628,"Pariser Leben: Finale I ""Auftritt des Brasilia...",Jacques Offenbach,
7,10000629,"Pariser Leben: Finale II ""Ich bin Witwe des Co...",Jacques Offenbach,
8,10000631,"Hoffmanns Erzählungen: Arie des Dapertutto ""Le...",Jacques Offenbach,
9,10000634,Die schöne Helena: Urteil des Paris,Jacques Offenbach,


In [20]:
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

likely_french_with_genre.to_csv(
    out_dir / "mb_likely_french_stage2_with_genre_long.tsv",
    sep="\t",
    index=False
)

likely_french_genre_agg.to_csv(
    out_dir / "mb_likely_french_stage2_with_genre_agg.tsv",
    sep="\t",
    index=False
)

print("Saved:")
print((out_dir / "mb_likely_french_stage2_with_genre_long.tsv").resolve())
print((out_dir / "mb_likely_french_stage2_with_genre_agg.tsv").resolve())

Saved:
/content/data/processed/mb_likely_french_stage2_with_genre_long.tsv
/content/data/processed/mb_likely_french_stage2_with_genre_agg.tsv


In [30]:
out_dir = Path("/content/drive/MyDrive/masters/data/dataset/processed")
out_dir.mkdir(parents=True, exist_ok=True)

likely_french_with_genre.to_csv(
    out_dir / "mb_likely_french_stage2_with_genre_long.tsv",
    sep="\t",
    index=False
)

likely_french_genre_agg.to_csv(
    out_dir / "mb_likely_french_stage2_with_genre_agg.tsv",
    sep="\t",
    index=False
)

candidates_stage1.to_csv(
    out_dir / "mb_candidates_stage1.tsv",
    sep="\t",
    index=False
)

france_country_subset.to_csv(
    out_dir / "mb_france_country_subset.tsv",
    sep="\t",
    index=False
)

likely_french_stage2.to_csv(
    out_dir / "mb_likely_french_stage2.tsv",
    sep="\t",
    index=False
)

print("Saved to Google Drive:")
for f in out_dir.glob("*.tsv"):
    print("-", f)

Saved to Google Drive:
- /content/drive/MyDrive/masters/data/dataset/processed/mb_likely_french_stage2_with_genre_long.tsv
- /content/drive/MyDrive/masters/data/dataset/processed/mb_likely_french_stage2_with_genre_agg.tsv
- /content/drive/MyDrive/masters/data/dataset/processed/mb_candidates_stage1.tsv
- /content/drive/MyDrive/masters/data/dataset/processed/mb_france_country_subset.tsv
- /content/drive/MyDrive/masters/data/dataset/processed/mb_likely_french_stage2.tsv
